# QUBO optimization: simulated annealing vs QAOA

QUBO instances solved via Ising lift. The cpu route runs simulated annealing; the cpu+sim route runs QAOA on the Ising Hamiltonian.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** combinatorial optimization. **Classical:** branch-and-bound, simulated annealing. **Quantum:** QAOA — alternating cost / mixer layers measured at a sweep of (γ, β) angles.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.core import types
from uniqx.core.enums import SpinChainModel
from uniqx.domains.physics.kernels import spin_chain_hamiltonian
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import apply_linear

n_spins = 3
dim = 1 << n_spins
# Random QUBO matrix lifts to an Ising Hamiltonian via the spin_chain
# kernel's QUBO model branch.
rng = np.random.default_rng(0)
Q = (rng.standard_normal((n_spins, n_spins)) * 0.5)
Q = 0.5 * (Q + Q.T)

@trace
def prog(q, state):
    h_flat = spin_chain_hamiltonian(n_spins=n_spins, model=SpinChainModel.QUBO,
                                     J=1.0, pbc=False, Q=q)
    h_mat = shape.reshape(h_flat, result_type=types.tensor("f64", [dim, dim]),
                           shape=[dim, dim])
    return apply_linear(h_mat, state)

state = np.ones(dim) / np.sqrt(dim)
module = prog(Q.tolist(), state.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
